In [15]:
!pip install line-bot-sdk

In [16]:
!pip install flask

In [59]:
from ultralytics import YOLO
import cv2
import pandas as pd
from flask import Flask, request

# 載入 json 標準函式庫，處理回傳的資料格式
import json

# 載入 LINE Message API 相關函式庫
from linebot import LineBotApi, WebhookHandler
from linebot.exceptions import InvalidSignatureError
from linebot.models import MessageEvent, TextMessage, TextSendMessage,StickerSendMessage,ImageSendMessage,VideoSendMessage

app = Flask(__name__)

@app.route("/", methods=['POST'])

def linebot():
    body = request.get_data(as_text=True)                    # 取得收到的訊息內容
    try:
        json_data = json.loads(body)                         # json 格式化訊息內容
        access_token = 'bhkNS9pWpH0nlris5s0kRFJY4a7apSovxGS9ZupF8fwQlpXKB+W0x2AdVj6b8Ke/u6oS5jCd8JHkNEQQDORoLVNofxyAAdWxFkKenzhItX7X0Xeph/cOiuyIEv2A5pJ0JpSLpcux7+ADNdTB1FJ9+gdB04t89/1O/w1cDnyilFU='
        secret = 'ca62bd85ff9d43487007533f15ae3abf'
        line_bot_api = LineBotApi(access_token)              # 確認 token 是否正確
        handler = WebhookHandler(secret)                     # 確認 secret 是否正確
        signature = request.headers['X-Line-Signature']      # 加入回傳的 headers
        handler.handle(body, signature)                      # 綁定訊息回傳的相關資訊
        tk = json_data['events'][0]['replyToken']            # 取得回傳訊息的 Token
        type = json_data['events'][0]['message']['type']     # 取得 LINe 收到的訊息類型
        if type=='text':
            msg = json_data['events'][0]['message']['text']  # 取得 LINE 收到的文字訊息
            recipy=pd.read_csv("data.csv")
            print(msg)                                       # 印出內容
            
            temp=(recipy['Search']==msg)
            temp2=recipy.loc[temp]
            feback_title="title="+str(temp2['title'].head(1))
            feback_ingredients="ingredients="+str(temp2['ingredients'].head(1))
            feback_directions="directions="+str(temp2['directions'].head(1))
            reply_text=[TextSendMessage(text=feback_title+feback_ingredients+feback_directions)]
            line_bot_api.reply_message(tk,reply_text)#回傳訊息
        elif type=='image':

            msgID=json_data['events'][0]['message']['id'] #取得訊息id
            message_content=line_bot_api.get_message_content(msgID) #根據訊息ID取得訊息內容
            file_path=f'static/{msgID}.jpg'
            with open(file_path,'wb') as fd:#在同樣的資料夾中建立以訊息ID為檔名的.jpg檔案
                fd.write(message_content.content)    #以二進位的方式寫入檔案
            img=cv2.imread(file_path)
            public_url='https://0bcb-122-100-89-165.ngrok-free.app/'+file_path  #取得影片公開URL
            model = YOLO(r"C:\Users\clair\OneDrive\Desktop\李佩臻_期中報告\runs\detect\train\weights\best.pt")
            results = model (img, save=False, show=False, hide_labels=True)
            total_type=[]
            for i in results:
                if len(i.boxes.xyxy)==0:
                    answer="好像沒東西..."
                    break;
                for j in i:
                        start_point=(int(j.boxes.xyxy.cpu().numpy()[0][0]),int(j.boxes.xyxy.cpu().numpy()[0][1]))
                        end_point=(int(j.boxes.xyxy.cpu().numpy()[0][2]), int(j.boxes.xyxy.cpu().numpy()[0][3]))
                        if int(j.boxes.cls.cpu().numpy()[0])==0:
                            total_type.append("tomato")

                        if int(j.boxes.cls.cpu().numpy()[0])==1:
                            total_type.append("egg")

                        if int(j.boxes.cls.cpu().numpy()[0])==2:
                            total_type.append("shrimp")

                        if int(j.boxes.cls.cpu().numpy()[0])==3:
                            total_type.append("cauliflower")

                        if int(j.boxes.cls.cpu().numpy()[0])==4:
                            total_type.append("onion")

                        if int(j.boxes.cls.cpu().numpy()[0])==5:
                            total_type.append("cabbage")

                        total_type=list(set(total_type))
                        answer=total_type
                        if len(total_type)<=1:
                            R=total_type[0]
                        else:
                            R=total_type[0]+'+'
                            for r in range(len(total_type)-1):
                                R+=total_type[r+1]
                        recipy=pd.read_csv("data.csv")
                        temp3=(recipy['Search']==str(R))
                        temp4=recipy.loc[temp3]
                        feback_title="title="+str(temp4['title'].head(1))
                        feback_ingredients="ingredients="+str(temp4['ingredients'].head(1))
                        feback_directions="directions="+str(temp4['directions'].head(1))
                        feback=feback_title+feback_ingredients+feback_directions
            reply=[    #回傳影片和文字訊息
                    TextSendMessage(text=str(answer)),
                    TextSendMessage(text=str(feback))
                ]                   #設定要回傳的訊息
        
            line_bot_api.reply_message(tk,reply)  #使用回復列表
    except:
        print(body)                                          # 如果發生錯誤，印出收到的內容
    return 'OK'                                              # 驗證 Webhook 使用，不能省略

if __name__ == "__main__":
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
C:\Users\clair\AppData\Local\Temp\ipykernel_10784\126688029.py:24: LineBotSdkDeprecatedIn30: Call to deprecated class LineBotApi. (Use v3 class; linebot.v3.<feature>. See https://github.com/line/line-bot-sdk-python/blob/master/README.rst for more details.) -- Deprecated since version 3.0.0.
  line_bot_api = LineBotApi(access_token)              # 確認 token 是否正確
C:\Users\clair\AppData\Local\Temp\ipykernel_10784\126688029.py:25: LineBotSdkDeprecatedIn30: Call to deprecated class WebhookHandler. (Use 'from linebot.v3.webhook import WebhookHandler' instead. See https://github.com/line/line-bot-sdk-python/blob/master/README.rst for more details.) -- Deprecated since version 3.0.0.
  handler = WebhookHandler(secret)                     # 確認 secret 是否正確
C:\Users\clair\AppData\Local\Temp\ipykernel_10784\126688029.py:45: LineBotSdkDeprecatedIn30: Call to deprecated method get_message_content. (Use 'from linebot.v3.messaging import Messagin

WARNING ⚠️ 'hide_labels' is deprecated and will be removed in 'ultralytics 8.2' in the future. Please use 'show_labels' instead.

0: 512x640 (no detections), 76.5ms
Speed: 2.0ms preprocess, 76.5ms inference, 0.0ms postprocess per image at shape (1, 3, 512, 640)


127.0.0.1 - - [26/Dec/2023 10:38:17] "POST / HTTP/1.1" 200 -


{"destination":"Uec95953d0ae8271c0dbfde68b18e1c84","events":[{"type":"message","message":{"type":"image","id":"487758953527116448","quoteToken":"6QvCNlnxBcoGWWFNZn67Gm-FiXjL5es3gs9AGclaD8ylerNSr_tt_vHfKXu54F-bh-ZrmJ5QKpbzlyikzs72QrjVwfCn6t8Mj236TUbMLaco5ZvXdSyM69xJEDK7GxJ6RYrGuGAHhpn_vqrKkyutFw","contentProvider":{"type":"line"}},"webhookEventId":"01HJHZSCX27CE1805KPTDKQV91","deliveryContext":{"isRedelivery":false},"timestamp":1703558296222,"source":{"type":"user","userId":"Ubc0d3bf8413c56d18d08ff397e2464ff"},"replyToken":"400b0ae1d5884ad89857d43569a8b927","mode":"active"}]}
